# Crop Disease Detection — Data Exploration

This notebook explores the PlantVillage dataset structure, class distribution,
and demonstrates the augmentation pipeline used during training.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})
print('Libraries loaded successfully.')

In [ ]:
# Load sample labels CSV
labels_path = Path('../dataset/sample_labels.csv')
df = pd.read_csv(labels_path)
print(f'Total samples: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
print()
print(df.head(10))

# Class distribution bar chart
class_counts = df.groupby(['label', 'split']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14, 5))
class_counts.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='black', linewidth=0.5)
ax.set_title('Class Distribution by Split', fontsize=14, fontweight='bold')
ax.set_xlabel('Disease Class', fontsize=11)
ax.set_ylabel('Image Count', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(title='Split', loc='upper right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Class imbalance analysis and weight computation
train_df = df[df['split'] == 'train']
class_counts_train = train_df['label'].value_counts().sort_index()

print('Training set class counts:')
print(class_counts_train)

# Compute inverse-frequency weights
total = class_counts_train.sum()
weights = total / class_counts_train
weights_normalised = weights / weights.sum()

print('\nClass weights (normalised):')
for cls, w in weights_normalised.items():
    print(f'  {cls:<25} {w:.4f}')

# Imbalance ratio
imbalance_ratio = class_counts_train.max() / class_counts_train.min()
print(f'\nImbalance ratio (max/min): {imbalance_ratio:.1f}x')

In [ ]:
# Augmentation pipeline demonstration
pipeline_steps = [
    'Resize(252, 252)                  # Slightly larger for random crop',
    'RandomHorizontalFlip(p=0.5)       # Mirror left-right',
    'RandomVerticalFlip(p=0.3)         # Mirror top-bottom',
    'RandomRotation(degrees=30)        # Rotate up to 30 degrees',
    'ColorJitter(b=0.2, c=0.2, s=0.2) # Brightness/contrast/saturation',
    'RandomCrop(224, 224)              # Final crop to target size',
    'ToTensor()                        # PIL Image -> FloatTensor [0,1]',
    'Normalize(ImageNet mean/std)      # Standardise channels',
]

print('Training Augmentation Pipeline:')
print('-' * 50)
for i, step in enumerate(pipeline_steps, 1):
    print(f'  {i}. {step}')
print()
print('Validation Pipeline (deterministic):')
print('-' * 50)
val_steps = ['Resize(224, 224)', 'CenterCrop(224)', 'ToTensor()', 'Normalize(ImageNet mean/std)']
for i, step in enumerate(val_steps, 1):
    print(f'  {i}. {step}')

In [ ]:
# Image normalisation concept with dummy array
dummy_image = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)

# Step 1: Convert to float [0, 1]
img_float = dummy_image.astype(np.float32) / 255.0

# Step 2: Apply ImageNet normalisation
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
img_normalised = (img_float - IMAGENET_MEAN) / IMAGENET_STD

print(f'Original  — min: {dummy_image.min()}, max: {dummy_image.max()}, dtype: {dummy_image.dtype}')
print(f'Float     — min: {img_float.min():.3f}, max: {img_float.max():.3f}, dtype: {img_float.dtype}')
print(f'Normalised— min: {img_normalised.min():.3f}, max: {img_normalised.max():.3f}, dtype: {img_normalised.dtype}')
print()
print('Per-channel stats after normalisation:')
for c, name in enumerate(['R', 'G', 'B']):
    ch = img_normalised[:, :, c]
    print(f'  {name}: mean={ch.mean():.4f}, std={ch.std():.4f}')

## Dataset Summary

| Property | Value |
|----------|-------|
| Total images | ~16,160 |
| Number of classes | 10 |
| Image format | RGB JPEG, 224×224 px |
| Train / Val / Test | 70% / 20% / 10% (stratified) |
| Most common class | Yellow_Leaf_Curl (~5,357) |
| Least common class | Mosaic_Virus (~373) |
| Imbalance ratio | ~14x |

**Key takeaways:**
- The dataset exhibits significant class imbalance (14x ratio).
- WeightedRandomSampler is used during training to mitigate this.
- ImageNet normalisation is applied for transfer-learning compatibility.
